# Day 17 — Attention Mechanics & Vision Transformers

**Agri-AI Bootcamp** · Phase 5 of 5: *Transformers & Ag-Agents* · Days 17–20

> Slice a leaf image into 16×16 patches, flatten into tokens and pass through a linear layer.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A Vision Transformer cuts an image into fixed patches, flattens each and projects it to an embedding, producing a sequence like words in a sentence. Self-attention lets every patch weigh every other patch. Cost grows with the number of tokens, so ViTs use more memory than small CNNs.

### 🌾 Agriculture example

A 224×224 leaf split into 16×16 patches gives 14×14 = 196 tokens, each with 16×16×3 = 768 numbers, then a linear layer maps them to the embedding size.

> **Watch out:** Image size must divide by patch size: 224 works, 226 doesn’t.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Data

In [ ]:
# ---- One leaf image as a tensor ---------------------------------------------
# Real images: PlantVillage.  Here a synthetic leaf with lesions, 224x224.
rng = np.random.default_rng(0)
size = 224
img = np.stack([rng.uniform(.10,.22,(size,size)), rng.uniform(.45,.68,(size,size)),
                rng.uniform(.10,.22,(size,size))]).astype(np.float32)
for _ in range(5):                                   # brown lesions
    cx, cy = rng.integers(30, size-30, 2); r = rng.integers(14, 26)
    yy, xx = np.ogrid[:size, :size]
    m = (xx-cx)**2 + (yy-cy)**2 <= r**2
    img[0][m], img[1][m], img[2][m] = .46, .28, .10
img = torch.tensor(img)
print("image:", tuple(img.shape))
plt.imshow(img.permute(1,2,0)); plt.axis("off"); plt.title("input leaf"); plt.show()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
img=torch.rand(3,224,224)                      # stand-in for a leaf photo
p=img.unfold(1,16,16).unfold(2,16,16)          # 3 x 14 x 14 x 16 x 16
tok=p.permute(1,2,0,3,4).reshape(196,-1)       # 196 tokens x 768
emb=nn.Linear(768,256)(tok)
```

## 5. Worked solution

In [ ]:
PATCH = 16
assert size % PATCH == 0, "image size must divide by patch size: 224 works, 226 does not"
G = size // PATCH                                   # 14 patches per side
print(f"{size}x{size} image / {PATCH}x{PATCH} patches -> {G}x{G} = {G*G} tokens")

# ---- Patchify with unfold ----------------------------------------------------
p = img.unfold(1, PATCH, PATCH).unfold(2, PATCH, PATCH)      # 3 x 14 x 14 x 16 x 16
print("after unfold:", tuple(p.shape))
tokens = p.permute(1, 2, 0, 3, 4).reshape(G*G, -1)           # 196 x 768
print("tokens      :", tuple(tokens.shape), f"({PATCH}*{PATCH}*3 = {PATCH*PATCH*3} numbers each)")

# the same thing in one line, the way real ViTs do it
proj = nn.Conv2d(3, 256, kernel_size=PATCH, stride=PATCH)    # patch embedding
emb_conv = proj(img.unsqueeze(0)).flatten(2).transpose(1, 2) # 1 x 196 x 256
print("conv patch-embedding:", tuple(emb_conv.shape))

# ---- Embed + class token + positions ----------------------------------------
DIM = 256
embed = nn.Linear(PATCH*PATCH*3, DIM)
x = embed(tokens).unsqueeze(0)                               # 1 x 196 x 256
cls = nn.Parameter(torch.zeros(1, 1, DIM))
pos = nn.Parameter(torch.randn(1, G*G + 1, DIM) * .02)
x = torch.cat([cls, x], 1) + pos                             # 1 x 197 x 256
print("\nsequence into the transformer:", tuple(x.shape), "(1 class token + 196 patches)")

# ---- Self-attention, written out --------------------------------------------
def attention(x, Wq, Wk, Wv):
    q, k, v = Wq(x), Wk(x), Wv(x)
    d = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / d ** .5             # every patch vs every patch
    w = scores.softmax(-1)                                   # rows sum to 1
    return w @ v, w

Wq, Wk, Wv = (nn.Linear(DIM, DIM) for _ in range(3))
out, w = attention(x, Wq, Wk, Wv)
print("output:", tuple(out.shape), "| attention map:", tuple(w.shape),
      "| each row sums to", w[0, 0].sum().item().__round__(3))

# ---- Why ViTs are memory-hungry ---------------------------------------------
print("\nattention cost grows with tokens^2:")
for ps in [32, 16, 8]:
    n = (size // ps) ** 2
    print(f"  patch {ps:>2} -> {n:>5} tokens -> attention matrix {n*n:>9,} entries")

# ---- The real thing, for comparison -----------------------------------------
blk = nn.TransformerEncoderLayer(DIM, nhead=8, dim_feedforward=512,
                                 batch_first=True, norm_first=True)
enc = nn.TransformerEncoder(blk, num_layers=4)
with torch.no_grad(): h = enc(x)
logits = nn.Linear(DIM, 2)(h[:, 0])                          # classify from CLS token
print("\n4-layer encoder out:", tuple(h.shape), "-> class logits", logits.detach().numpy().round(3))
print("encoder params:", f"{sum(p.numel() for p in enc.parameters()):,}")

# ---- Where does one patch look? ---------------------------------------------
probe = 100                                                   # token index to inspect
amap = w[0, probe, 1:].reshape(G, G).detach()
fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
ax[0].imshow(img.permute(1,2,0)); ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(amap, cmap="magma"); ax[1].set_title(f"attention of patch {probe}")
ax[1].axis("off"); plt.tight_layout(); plt.show()
print("Untrained weights -> a near-uniform map. Training is what makes it meaningful.")

## 6. Your turn

1. Set PATCH = 8 and PATCH = 32 and record token count and attention-matrix size.
2. Try size = 226 and read the assertion error — that is the divisibility rule.
3. Split attention into 4 heads by reshaping to (heads, tokens, dim/heads).

In [ ]:
# TODO 1: patch-size sweep
for ps in [8, 16, 32]:
    g = size // ps
    t = img.unfold(1, ps, ps).unfold(2, ps, ps).permute(1,2,0,3,4).reshape(g*g, -1)
    print(f"patch {ps:>2}: {t.shape[0]:>4} tokens of {t.shape[1]:>4} numbers")

# TODO 2: non-divisible size
# bad = torch.rand(3, 226, 226); bad.unfold(1, 16, 16).unfold(2, 16, 16).shape  # loses pixels

# TODO 3: multi-head attention by hand
def mha(x, Wq, Wk, Wv, heads=4):
    B, N, D = x.shape; d = D // heads
    q = Wq(x).view(B, N, heads, d).transpose(1, 2)
    k = Wk(x).view(B, N, heads, d).transpose(1, 2)
    v = Wv(x).view(B, N, heads, d).transpose(1, 2)
    a = (q @ k.transpose(-2,-1) / d**.5).softmax(-1)
    return (a @ v).transpose(1, 2).reshape(B, N, D)
print("multi-head out:", tuple(mha(x, Wq, Wk, Wv).shape))

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)

**Course material:** Jay Alammar: The Illustrated Transformer

---
**Next:** Day 18 — Zero-Shot Ag-Classification with CLIP